In [1]:
from langchain_community.retrievers import WikipediaRetriever

C:\Users\Shravan\AppData\Local\Temp\ipykernel_25140\2879121110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [2]:
retriever = WikipediaRetriever(
    lang='en',
    top_k_results=1
)

In [3]:
docs = retriever.invoke('the geopolitical history of India from perspective of china')# invoke method denotes that this retriever is a runnable
docs

[Document(metadata={'title': 'Geopolitical economy', 'summary': 'Geopolitical economy is a contemporary Marxist approach to understanding the capitalist world historically. It was proposed by Radhika Desai in her Geopolitical Economy: After US Hegemony, Globalization and Empire as a critique of contemporary mainstream theories of International political economy (IPE) and International relations (IR). Geopolitical economy\'s critique rests on a rejection of orthodox views of the world economy as a seamless whole, united either by markets or by a single leading state, as in free market, free trade "globalization" and "hegemony" theories respectively. Instead, geopolitical economy emphasizes the interplay of political entities, namely, states, in the development of capitalism by going back to classical political economy and to the Marxist theories of imperialism, which geopolitical economy argues should be considered the first theories of international relations.', 'source': 'https://en.w

In [4]:
for doc in docs:
    page_content = doc.page_content

print('Page Content')
print(page_content)

Page Content
Geopolitical economy is a contemporary Marxist approach to understanding the capitalist world historically. It was proposed by Radhika Desai in her Geopolitical Economy: After US Hegemony, Globalization and Empire as a critique of contemporary mainstream theories of International political economy (IPE) and International relations (IR). Geopolitical economy's critique rests on a rejection of orthodox views of the world economy as a seamless whole, united either by markets or by a single leading state, as in free market, free trade "globalization" and "hegemony" theories respectively. Instead, geopolitical economy emphasizes the interplay of political entities, namely, states, in the development of capitalism by going back to classical political economy and to the Marxist theories of imperialism, which geopolitical economy argues should be considered the first theories of international relations.


== Background ==
The terms "geo-political economy" and "geo-politics’ had be

#### vector store retriever 

A vector store retriever in Langchain is the most common type of retriever that lets you search and fetch documents from a vector store based on semantic similarity using vector embeddings 

1. you store documents in vector store(like pinecone,chroma)
2. each document is converted in a dense vector using a embedding model
3. when the user enters a query
    - it's also turned in a vector
    - the retriever compares the query vector with the stored vectors
    - it retrieves the top-k most  similar ones

In [5]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

c:\Users\Shravan\Music\starting_genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
documents = [
    Document(page_content='Langchain helps developers build LLM applications easily'),
    Document(page_content='Chroma is a vector database optimized for LLM based search'),
    Document(page_content='Embeddings convert text into high dimensional vectors'),
    Document(page_content='Huggingface provides access to various opensource models')
]

In [7]:
load_dotenv()
embedding_model = HuggingFaceEndpointEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2"
)

In [8]:
vector_store = Chroma(
    persist_directory='chroma_data',
    embedding_function=embedding_model,
    collection_name='wikipedia_retriever_demo'
)

In [9]:
vector_store.add_documents(documents)

['b5d2e906-49c9-40fd-a80f-86777a142ac3',
 '80738ee7-ddbe-4f7b-91b7-3d94a5ca93bf',
 'd0c1398c-a197-46e6-809e-ebbda6d9bd2a',
 '19fbc307-2339-4b14-8dcd-4c0a10fc2bf2']

In [10]:
## convert a vector store in a retriever 

retriever = vector_store.as_retriever(search_kwargs={"k":2})


In [11]:
query = 'what is chroma used for'

result = retriever.invoke(query)
result

[Document(id='2f15969b-67cb-4f8d-b7b8-4ec2e37afb4f', metadata={}, page_content='Chroma is a vector database optimized for LLM based search'),
 Document(id='80738ee7-ddbe-4f7b-91b7-3d94a5ca93bf', metadata={}, page_content='Chroma is a vector database optimized for LLM based search')]

In [12]:
for i,doc in enumerate(result):
    print('Result',i+1)
    print(doc.page_content)


Result 1
Chroma is a vector database optimized for LLM based search
Result 2
Chroma is a vector database optimized for LLM based search


we could have used the similarity_search() function of vector store 

main differences between the similarity_search() and retriever

 Feature              | similarity_search()                  | Retriever

 What it is           | Direct vector store method            | Wrapper around vector store
 Used for             | Manual searching                      | RAG chains and pipelines
 Input                | Query string                          | Query string
 Output               | Documents                             | Documents
 Supports k           | Yes                                   | Yes, through search_kwargs
 Easy to use in chains| Less preferred                        | Preferred
 Example              | vector_store.similarity_search(       | retriever.invoke(query)
                      |     query, k=3                        |
                      | )                                     |



##### Maximal Marginal Relevance(MMR)

MMr is information retrieval stratergy used to reduce the redundancy in the 
retrieved results while maintaining the high relevance to the query

how can we pick the results that are not only relevant to the query but also different from each other
MMR is information retrieval  algorithm designed to reduce redundancy in the retriever results while 
maintaining high relevance to the query 

For example, if the query is “What is LangChain?”, normal similarity search may return three chunks that all say LangChain is used to build LLM applications. These chunks are relevant, but repetitive. MMR instead may return one chunk explaining what LangChain is, one chunk about chains and prompts, and one chunk about retrievers or vector stores. So MMR gives the LLM more useful and diverse context, not just similar repeated information.

In [13]:
sample_docs = [
    Document(page_content ='Langchain makes it easy to work with LLMs'),
    Document(page_content ='Langchain is used to build LLM based applications'),
    Document(page_content='Chroma is used to store and search document embeddings'),
    Document(page_content='Embeddings are vector representation of text.'),
    Document(page_content='MMR helps you get diverse results when doing similarity search'),
    Document(page_content='Langchain supports Chroma,FAiss,pine cone and more')
]

In [14]:
from langchain_community.vectorstores import FAISS

In [15]:
vector_store = FAISS.from_documents(
    documents=sample_docs,
    embedding = embedding_model,
)

In [34]:
retriever = vector_store.as_retriever(
    search_type = 'mmr', # defining the technique
    search_kwargs={"k": 3, "lambda_mult": 0} # k = top results, lambda mult = relevance diversity balance
)# lamdba_mult gives result between 0 and 1, when it is set to 0 it gives very diverse results, and when 1 it behaves like normal similarity search

In [35]:
result = retriever.invoke('What is Langchain')

In [36]:
for i , doc in enumerate(result):
    print('Result',i+1)
    print(doc.page_content) # relevant yet diverse results 
    print('-'*20)

Result 1
Langchain supports Chroma,FAiss,pine cone and more
--------------------
Result 2
MMR helps you get diverse results when doing similarity search
--------------------
Result 3
Embeddings are vector representation of text.
--------------------


##### Multi query retriever

sometimes a single query might not capture all the ways information is phared in your documents
Example 

query = 'How can i stay healthy'

The above query could mean that - 
- What should I eat?
- How often should I excercise?
- How can I  manage stress?

A simple similarity search might miss documents that talk about those things but don't use the word 'healthy'


In [39]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [58]:
from langchain_core.documents import Document

all_docs = [
    Document(
        page_content="Machine learning is a field of artificial intelligence where computers learn patterns from data and use those patterns to make predictions or decisions.",
        metadata={"source": "ml_doc", "category": "ai"}
    ),
    Document(
        page_content="Supervised learning trains a model using labeled examples, where each input has a correct output already provided.",
        metadata={"source": "supervised_learning_doc", "category": "ai"}
    ),
    Document(
        page_content="Training data is the collection of examples used to teach a machine learning model how to recognize patterns.",
        metadata={"source": "training_data_doc", "category": "ai"}
    ),
    Document(
        page_content="A database stores and manages data so that applications can insert, update, search, and retrieve information efficiently.",
        metadata={"source": "database_doc", "category": "database"}
    ),
    Document(
        page_content="A web application is software that runs in a browser and allows users to interact with backend services through the internet.",
        metadata={"source": "web_app_doc", "category": "web"}
    ),
]

In [59]:
## define the vector store 
faiss_vector_store = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

In [60]:
## create a normal retriever -> which performs search based on similarity 

similarity_retriever = faiss_vector_store.as_retriever(
    search_type ="similarity", # when using chroma no need to mention explicitly, default is similarity
    search_kwargs = {"k":5}
)

In [61]:
## create a multi query retriever which generate multiple different versions of that query.
## These different query versions are used to search the vector database from multiple angles,
## so it can find more relevant documents than a normal retriever.

from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id = 'deepseek-ai/DeepSeek-V4-Pro',
    task = 'text-generation'
)

llm_model = ChatHuggingFace(llm=llm)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = faiss_vector_store.as_retriever(search_kwargs = {"k":5}),
    llm = llm_model
)

In [62]:
## Query 
query = "How does it learn?"

## retrieve results

similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [64]:
# using normal retriever
for i,j in enumerate(similarity_results):
    print(f'Result {i+1}')
    print(j.page_content) # we would get 5 docs because k was set to 5
    print('-'*40)

Result 1
Supervised learning trains a model using labeled examples, where each input has a correct output already provided.
----------------------------------------
Result 2
Machine learning is a field of artificial intelligence where computers learn patterns from data and use those patterns to make predictions or decisions.
----------------------------------------
Result 3
Training data is the collection of examples used to teach a machine learning model how to recognize patterns.
----------------------------------------
Result 4
A database stores and manages data so that applications can insert, update, search, and retrieve information efficiently.
----------------------------------------
Result 5
A web application is software that runs in a browser and allows users to interact with backend services through the internet.
----------------------------------------


In [57]:
for i,docu in enumerate(multiquery_results):
    print(f'Result {i+1}')
    print(docu.page_content)
    print('-'*40)

Result 1
Machine learning allows computers to learn patterns from data and make predictions without being explicitly programmed.
----------------------------------------
Result 2
MLflow is a tool used to track machine learning experiments, manage models, and organize the model development lifecycle.
----------------------------------------
Result 3
Deep learning is a subset of machine learning that uses neural networks with many layers to learn complex patterns from large datasets.
----------------------------------------
Result 4
Logistic Regression is a supervised machine learning algorithm used mainly for classification problems such as spam detection and sentiment analysis.
----------------------------------------
Result 5
Natural Language Processing helps computers understand, process, and generate human language such as text and speech.
----------------------------------------
Result 6
MySQL is a relational database management system that stores structured data using tables, rows

##### Contextual compression retriever
The contextual compression retriever in langchain is an advanced retriever that improves retrieval
quality by compressing documents after retrieval, keeping only the relevant context based on user 's query 

Example: query -> What is photosynthesis

"The grand canyon is a famous natural site.Photosynthesis is how plants convert light into energy.Many tourist visit every year".

problem 
- The retriever returns the entire paragraph
- Only one sentence is actually relevant to the query 
- the rest of irrlevant noise that waste the window context and may confuse the llm


In [65]:
## how does it work 
"""
Query
  ↓
Retriever with k=2
  ↓
Top 2 documents: doc1, doc2
  ↓
Send query + doc1 + doc2 to prompt
  ↓
LLM removes irrelevant parts from each document
  ↓
Return cleaned doc1 and cleaned doc2
"""

'\nQuery\n  ↓\nRetriever with k=2\n  ↓\nTop 2 documents: doc1, doc2\n  ↓\nSend query + doc1 + doc2 to prompt\n  ↓\nLLM removes irrelevant parts from each document\n  ↓\nReturn cleaned doc1 and cleaned doc2\n'

In [68]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [69]:
input_docs = [
    Document(
        page_content="""
             The Grand canyon is one of the most visited natural wonders in the world.
             Photosynthesis is the process by which green plants convert sunlight into energy.
             Millions of tourist travel to see it every year. The rocks date back millions of years.
             """,
        metadata={"source": "doc1"},
    ),
    Document(
        page_content="""
             In medival Europe, castles were built primarily for defense.
             The chlorophyll in plant cells captures the sunlight during photosynthesis.
             Knights wore armour made of metal. Siege weapons were often used to breach castle walls.
             """,
        metadata={"source": "doc2"},
    ),
    Document(
        page_content="""
             Basketball was invented by Dr. James Naismith in late 19th century.
             It was originally played with a soccer ball and peach baskets. NBA is now a global league.
             """,
        metadata={"source": "doc3"},
    ),
    Document(
        page_content="""
             History of cinema began in the late 1800s. Silent films wree the earliest form.
             Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
             Modern filmaking involves complex CGI and sound design.
             """,
        metadata={"source": "doc3"},
    )
]

In [70]:
vector_store_context = FAISS.from_documents(
    documents=input_docs,
    embedding=embedding_model
)

In [71]:
base_retriever = vector_store_context.as_retriever(search_kwargs={"k": 5})

In [72]:
## setup the compressor using an LLM 

compressor = LLMChainExtractor.from_llm(llm_model)

In [73]:
## create the contextual compression retriever 

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
) # retriever will retrieve the docs and the compressor will improve the quality

In [74]:
## query the retriever 

query = 'What is photosynthesis'
compressor_result = compression_retriever.invoke(query)

for i,doc in enumerate(compressor_result):
    print('Result',i+1)
    print(doc.page_content)
    print('-'*30)

Result 1
Photosynthesis is the process by which green plants convert sunlight into energy.
------------------------------
Result 2
The chlorophyll in plant cells captures the sunlight during photosynthesis.
------------------------------
Result 3
Photosynthesis does not occur in animal cells.
------------------------------
